# Decompose Direct xG And Downstream Chain Value

This notebook tests whether outside-origin shots are better evaluated as possession-chain decisions rather than terminal shot events.

The core question is:

> When an outside shot is treated as the start of a possession chain, how much value appears beyond the origin-shot xG, and where is that added value concentrated?

Notebook 11 compares:

- first evaluated xG
- maximum xG within the chain
- summed chain xG from the processed chain dataset
- observed chain goal rate

Independent chain xG is not computed here because chance-level chain membership was not saved as a processed output. It can be added later by reconstructing chain membership.

## 1. Setup

Load libraries, define paths, and configure display options.

In [1]:
# Import libraries used for chain-value decomposition

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data exists:", DATA_PROCESSED.exists())
print("Figures dir:", FIGURES_DIR)

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
Processed data exists: True
Figures dir: c:\Users\rinal\hockey-analytics\outside-shot-value\figures


In [3]:
# Configure pandas display options for readable notebook tables

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

## 2. Inspect Processed Data Files

Identify which processed files are available.

This matters because full chain-value decomposition requires a chance-level chain-membership table. If that table was not saved previously, we will reconstruct it from the available processed data in a later section.

In [4]:
# List processed parquet files available to this notebook

processed_files = sorted(DATA_PROCESSED.glob("*.parquet"))

processed_file_inventory = pd.DataFrame(
    {
        "file_name": [p.name for p in processed_files],
        "path": [str(p.relative_to(PROJECT_ROOT)) for p in processed_files],
    }
)

processed_file_inventory

,file_name,path
0,origin_shot_sequences.parquet,data\processed\origin_shot_sequences.parquet
1,origin_shot_sequences_context_labeled.parquet,data\processed\origin_shot_sequences_context_l...
2,origin_shot_sequences_labeled.parquet,data\processed\origin_shot_sequences_labeled.p...
3,origin_shot_sequences_with_tracking.parquet,data\processed\origin_shot_sequences_with_trac...
4,shot_value_base.parquet,data\processed\shot_value_base.parquet


In [5]:
# Load the context-labeled chain dataset from Notebook 08

chains = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences_context_labeled.parquet")

print("chains:", chains.shape)
chains.head()

chains: (48673, 111)


,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,n_chances_in_chain,anchor_origin_event_id,anchor_origin_period_time,anchor_origin_location,anchor_first_chance_event_id,anchor_first_chance_time,anchor_first_event_type,anchor_first_chance_location,first_evaluated_chance_event_id,first_evaluated_chance_time,first_evaluated_event_type,first_evaluated_location,first_evaluated_xg,chain_max_xg,chain_sum_xg,has_deflection,n_deflections,first_deflection_event_id,first_deflection_time,deflection_max_xg,deflection_sum_xg,has_followup_chance,n_followup_chances,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg,chain_goal,chain_goal_event_id,chain_goal_time,chain_goal_player_id,chain_goal_player_name,chain_any_goal_within_2s_flag,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked,origin_tracking_available,origin_tracking_error_flag,origin_tracking_completeness_bucket,home_team,away_team,home_team_id,away_team_id,team_side,period_time_start,game_stint,period_time_end,n_home_skaters,n_away_skaters,is_home_net_empty,is_away_net_empty,home_score,away_score,origin_time_within_stint,is_5v5,team_n_skaters,opponent_n_skaters,team_net_empty,opponent_net_empty,raw_anchor_origin_period_time,anchor_origin_event_type_raw,anchor_origin_event_team,anchor_origin_x_adj,anchor_origin_y_adj,is_offensive_zone_origin,is_behind_offensive_blue_line_origin,chain_team_name,prior_faceoff_event_id,prior_faceoff_time,prior_faceoff_team,prior_faceoff_x_adj_for_event_team,prior_faceoff_y_adj_for_event_team,seconds_since_prior_faceoff,prior_faceoff_x_adj_for_chain_team,prior_faceoff_y_adj_for_chain_team,has_prior_faceoff_in_sequence,prior_faceoff_within_5s,prior_faceoff_in_offensive_zone_for_chain,oz_faceoff_context_5s,first_same_team_event_time_in_sequence,n_same_team_events_in_sequence,seconds_since_first_same_team_event_in_sequence,early_same_team_sequence_5s_diagnostic,settled_sequence_gt5s_diagnostic,entry_proxy_event_id,entry_proxy_time,entry_proxy_event_type,entry_proxy_x_adj_for_chain_team,entry_proxy_y_adj_for_chain_team,seconds_since_entry_proxy,has_entry_proxy_in_sequence,entry_proxy_within_5s,oz_entry_context_5s_proxy,settled_offensive_zone_context_proxy,shot_context
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,591.730000,591.730000,0.000000,1,565,591.730000,outside,565,591.730000,shot,outside,565,591.730000,shot,outside,0.040997,0.040997,0.040997,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,8,7,1,4,3,1,2,0,True,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,away,589.030000,64.000000,605.030000,4.000000,4.000000,False,False,1.000000,1.000000,True,False,4.000000,4.000000,False,False,591.730000000,shot,CLE,32.889107,25.397057,True,False,CLE,558.000000,589.030000,CLE,69.104380,21.376137,2.700000,69.104380,21.376137,True,True,True,True,589.030000,20,2.700000,True,False,560.000000,589.100000,pass,69.100876,23.385292,2.630000,True,True,True,False,oz_faceoff_5s
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,722.400000,722.400000,0.000000,1,687,722.400000,outside,687,722.400000,shot,outside,687,722.400000,shot,outside,0.015540,0.015540,0.015540,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,10,7,3,4,3,1,2,0,True,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f

In [6]:
# Inspect columns in the context-labeled chain dataset

chains.columns.tolist()

['chain_id',
 'game_id',
 'period',
 'sequence_id',
 'team_id',
 'raw_first_chance_team_id',
 'n_team_id_overrides_for_chain',
 'chain_start_time',
 'chain_end_time',
 'chain_duration_seconds',
 'n_chances_in_chain',
 'anchor_origin_event_id',
 'anchor_origin_period_time',
 'anchor_origin_location',
 'anchor_first_chance_event_id',
 'anchor_first_chance_time',
 'anchor_first_event_type',
 'anchor_first_chance_location',
 'first_evaluated_chance_event_id',
 'first_evaluated_chance_time',
 'first_evaluated_event_type',
 'first_evaluated_location',
 'first_evaluated_xg',
 'chain_max_xg',
 'chain_sum_xg',
 'has_deflection',
 'n_deflections',
 'first_deflection_event_id',
 'first_deflection_time',
 'deflection_max_xg',
 'deflection_sum_xg',
 'has_followup_chance',
 'n_followup_chances',
 'first_followup_event_id',
 'first_followup_time',
 'followup_max_xg',
 'followup_sum_xg',
 'chain_goal',
 'chain_goal_event_id',
 'chain_goal_time',
 'chain_goal_player_id',
 'chain_goal_player_name',
 'ch

## 3. Validate Chain-Level Inputs

Confirm the one-row-per-chain structure and identify the chain-value fields already available.

In [7]:
# Check required chain-level fields for Notebook 11

required_chain_columns = [
    "chain_id",
    "game_id",
    "period",
    "sequence_id",
    "team_id",
    "is_5v5",
    "anchor_origin_location",
    "is_offensive_zone_origin",
    "is_behind_offensive_blue_line_origin",
    "shot_context",
    "first_evaluated_xg",
    "chain_max_xg",
    "followup_max_xg",
    "has_followup_chance",
    "has_deflection",
    "chain_goal",
    "origin_tracking_available",
    "origin_tracking_error_flag",
    "observed_origin_attackers_in_slot",
    "observed_origin_defenders_in_slot",
]

missing_required_chain_columns = [
    c for c in required_chain_columns
    if c not in chains.columns
]

missing_required_chain_columns

[]

In [8]:
# Validate the chain-level input dataset

chain_input_validation = {
    "rows": len(chains),
    "unique_chain_ids": chains["chain_id"].nunique(),
    "duplicate_chain_ids": chains.duplicated("chain_id").sum(),
    "missing_first_evaluated_xg": chains["first_evaluated_xg"].isna().sum(),
    "missing_chain_max_xg": chains["chain_max_xg"].isna().sum(),
    "missing_chain_goal": chains["chain_goal"].isna().sum(),
    "missing_shot_context": chains["shot_context"].isna().sum(),
}

chain_input_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_first_evaluated_xg': np.int64(0),
 'missing_chain_max_xg': np.int64(0),
 'missing_chain_goal': np.int64(0),
 'missing_shot_context': np.int64(0)}

In [9]:
# Identify whether chain-sum or chain-independent xG fields already exist

existing_chain_value_columns = [
    c for c in chains.columns
    if "chain" in c.lower() and "xg" in c.lower()
]

existing_chain_value_columns

['chain_max_xg', 'chain_sum_xg']

## 4. Validate Available Chain-Value Fields

The processed chain dataset already contains `chain_max_xg` and `chain_sum_xg`.

Because no chain-member table is currently saved, this notebook uses:

- `first_evaluated_xg` as origin-shot value
- `chain_max_xg` as conservative downstream best-chance value
- `chain_sum_xg` as simple aggregate chain value
- `chain_goal` as observed empirical outcome

Independent chain xG can be added later by reconstructing chance-level chain membership.

In [10]:
# Validate existing chain-value fields before decomposition

chain_value_validation = {
    "rows": len(chains),
    "missing_first_evaluated_xg": chains["first_evaluated_xg"].isna().sum(),
    "missing_chain_max_xg": chains["chain_max_xg"].isna().sum(),
    "missing_chain_sum_xg": chains["chain_sum_xg"].isna().sum(),
    "chain_max_less_than_first_count": (
        chains["chain_max_xg"] < chains["first_evaluated_xg"]
    ).sum(),
    "chain_sum_less_than_chain_max_count": (
        chains["chain_sum_xg"] < chains["chain_max_xg"]
    ).sum(),
    "chain_sum_less_than_first_count": (
        chains["chain_sum_xg"] < chains["first_evaluated_xg"]
    ).sum(),
    "chain_sum_equals_chain_max_count": (
        np.isclose(chains["chain_sum_xg"], chains["chain_max_xg"])
    ).sum(),
}

chain_value_validation

{'rows': 48673,
 'missing_first_evaluated_xg': np.int64(0),
 'missing_chain_max_xg': np.int64(0),
 'missing_chain_sum_xg': np.int64(0),
 'chain_max_less_than_first_count': np.int64(0),
 'chain_sum_less_than_chain_max_count': np.int64(0),
 'chain_sum_less_than_first_count': np.int64(0),
 'chain_sum_equals_chain_max_count': np.int64(43960)}

In [11]:
# Create direct-vs-chain value fields

chains = chains.copy()

chains["upgrade_max_minus_first_xg"] = (
    chains["chain_max_xg"] - chains["first_evaluated_xg"]
)

chains["upgrade_sum_minus_first_xg"] = (
    chains["chain_sum_xg"] - chains["first_evaluated_xg"]
)

chains["chain_sum_minus_max_xg"] = (
    chains["chain_sum_xg"] - chains["chain_max_xg"]
)

chains[
    [
        "first_evaluated_xg",
        "chain_max_xg",
        "chain_sum_xg",
        "upgrade_max_minus_first_xg",
        "upgrade_sum_minus_first_xg",
        "chain_sum_minus_max_xg",
    ]
].describe()

,first_evaluated_xg,chain_max_xg,chain_sum_xg,upgrade_max_minus_first_xg,upgrade_sum_minus_first_xg,chain_sum_minus_max_xg
count,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000,48673.000000
mean,0.045373,0.053214,0.057150,0.007841,0.011777,0.003936
std,0.078502,0.090148,0.101723,0.044966,0.060809,0.024804
min,0.000610,0.000610,0.000610,0.000000,0.000000,0.000000
25%,0.004025,0.004435,0.004502,0.000000,0.000000,0.000000
50%,0.015797,0.018683,0.018949,0.000000,0.000000,0.000000
75%,0.051575,0.061151,0.063316,0.000000,0.000000,0.000000
max,0.938368,0.938368,1.462747,0.825043,1.431566,0.892610


## 5. Define Analysis Cohort And Slot-Support Bins

Use the same clean cohort as Notebooks 09 and 10:

- true 5v5
- outside-origin
- offensive-zone origin
- tracking available
- no tracking error flag

Then recreate observed attacking slot-support bins.

In [12]:
# Create readable boolean masks for the clean outside-origin analysis cohort

is_5v5 = chains["is_5v5"]
is_outside_origin = chains["anchor_origin_location"].eq("outside")
is_offensive_zone_origin = chains["is_offensive_zone_origin"]
has_tracking_available = chains["origin_tracking_available"]
has_no_tracking_error = ~chains["origin_tracking_error_flag"]

analysis_mask = (
    is_5v5
    & is_outside_origin
    & is_offensive_zone_origin
    & has_tracking_available
    & has_no_tracking_error
)

analysis = chains[analysis_mask].copy()

print("analysis:", analysis.shape)
analysis.head()

analysis: (21536, 114)


,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,n_chances_in_chain,anchor_origin_event_id,anchor_origin_period_time,anchor_origin_location,anchor_first_chance_event_id,anchor_first_chance_time,anchor_first_event_type,anchor_first_chance_location,first_evaluated_chance_event_id,first_evaluated_chance_time,first_evaluated_event_type,first_evaluated_location,first_evaluated_xg,chain_max_xg,chain_sum_xg,has_deflection,n_deflections,first_deflection_event_id,first_deflection_time,deflection_max_xg,deflection_sum_xg,has_followup_chance,n_followup_chances,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg,chain_goal,chain_goal_event_id,chain_goal_time,chain_goal_player_id,chain_goal_player_name,chain_any_goal_within_2s_flag,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked,origin_tracking_available,origin_tracking_error_flag,origin_tracking_completeness_bucket,home_team,away_team,home_team_id,away_team_id,team_side,period_time_start,game_stint,period_time_end,n_home_skaters,n_away_skaters,is_home_net_empty,is_away_net_empty,home_score,away_score,origin_time_within_stint,is_5v5,team_n_skaters,opponent_n_skaters,team_net_empty,opponent_net_empty,raw_anchor_origin_period_time,anchor_origin_event_type_raw,anchor_origin_event_team,anchor_origin_x_adj,anchor_origin_y_adj,is_offensive_zone_origin,is_behind_offensive_blue_line_origin,chain_team_name,prior_faceoff_event_id,prior_faceoff_time,prior_faceoff_team,prior_faceoff_x_adj_for_event_team,prior_faceoff_y_adj_for_event_team,seconds_since_prior_faceoff,prior_faceoff_x_adj_for_chain_team,prior_faceoff_y_adj_for_chain_team,has_prior_faceoff_in_sequence,prior_faceoff_within_5s,prior_faceoff_in_offensive_zone_for_chain,oz_faceoff_context_5s,first_same_team_event_time_in_sequence,n_same_team_events_in_sequence,seconds_since_first_same_team_event_in_sequence,early_same_team_sequence_5s_diagnostic,settled_sequence_gt5s_diagnostic,entry_proxy_event_id,entry_proxy_time,entry_proxy_event_type,entry_proxy_x_adj_for_chain_team,entry_proxy_y_adj_for_chain_team,seconds_since_entry_proxy,has_entry_proxy_in_sequence,entry_proxy_within_5s,oz_entry_context_5s_proxy,settled_offensive_zone_context_proxy,shot_context,upgrade_max_minus_first_xg,upgrade_sum_minus_first_xg,chain_sum_minus_max_xg
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,722.400000,722.400000,0.000000,1,687,722.400000,outside,687,722.400000,shot,outside,687,722.400000,shot,outside,0.015540,0.015540,0.015540,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,10,7,3,4,3,1,2,0,True,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,home,719.030000,80.000000,725.030000,5.000000,5.000000,False,False,1.000000,1.000000,True,True,5.000000,5.000000,False,False,722.400000000,shot,GR,52.107956,-8.799999,True,False,GR,589.000000,620.030000,CLE,-19.413269,20.873200,102.370000,19.413269,-20.873200,True,False,False,False,620.030000,89,102.370000,False,True,598.000000,637.530000,carry,25.955010,-36.461760,84.870000,True,False,False,True,settled_offensive_zone_proxy,0.000000,0.000000,0.000000
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,923.800000,923.800000,0.000000,1,882,923.800000,outside,882,923.800000,shot,outside,882,923.800000,shot,outside,0.010167,0.010167,0.010167,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,F

In [13]:
# Recreate observed attacking and defending slot-support bins

def cap_slot_count(value):
    if pd.isna(value):
        return pd.NA

    value = int(value)

    if value >= 3:
        return "3+"

    return str(value)


slot_bin_order = ["0", "1", "2", "3+"]

analysis["observed_attackers_slot_bin"] = (
    analysis["observed_origin_attackers_in_slot"].apply(cap_slot_count)
)

analysis["observed_defenders_slot_bin"] = (
    analysis["observed_origin_defenders_in_slot"].apply(cap_slot_count)
)

analysis["observed_attackers_slot_bin"] = pd.Categorical(
    analysis["observed_attackers_slot_bin"],
    categories=slot_bin_order,
    ordered=True,
)

analysis["observed_defenders_slot_bin"] = pd.Categorical(
    analysis["observed_defenders_slot_bin"],
    categories=slot_bin_order,
    ordered=True,
)

analysis[
    [
        "observed_origin_attackers_in_slot",
        "observed_attackers_slot_bin",
        "observed_origin_defenders_in_slot",
        "observed_defenders_slot_bin",
    ]
].head()

,observed_origin_attackers_in_slot,observed_attackers_slot_bin,observed_origin_defenders_in_slot,observed_defenders_slot_bin
1,1,1,2,2
4,0,0,1,1
8,2,2,1,1
10,1,1,2,2
13,1,1,2,2


In [14]:
# Validate analysis cohort and slot-support bins

analysis_validation = {
    "analysis_rows": len(analysis),
    "analysis_unique_chain_ids": analysis["chain_id"].nunique(),
    "analysis_duplicate_chain_ids": analysis.duplicated("chain_id").sum(),
    "behind_blue_line_rows": int(analysis["is_behind_offensive_blue_line_origin"].sum()),
    "missing_observed_attackers_slot_bin": analysis["observed_attackers_slot_bin"].isna().sum(),
    "slot_bin_counts": analysis["observed_attackers_slot_bin"].value_counts(dropna=False).sort_index().to_dict(),
    "context_counts": analysis["shot_context"].value_counts(dropna=False).to_dict(),
}

analysis_validation

{'analysis_rows': 21536,
 'analysis_unique_chain_ids': 21536,
 'analysis_duplicate_chain_ids': np.int64(0),
 'behind_blue_line_rows': 0,
 'missing_observed_attackers_slot_bin': np.int64(0),
 'slot_bin_counts': {'0': 8280, '1': 8202, '2': 4427, '3+': 627},
 'context_counts': {'settled_offensive_zone_proxy': 15387,
  'oz_entry_5s_proxy': 4020,
  'oz_faceoff_5s': 2115,
  'other_or_unknown': 14}}

## 6. Overall Direct-vs-Chain Value Decomposition

Compare origin-shot xG to downstream chain-value proxies for all clean 5v5 outside-origin offensive-zone chains.

This establishes the baseline difference between treating an outside shot as terminal and treating it as part of a possession chain.

In [15]:
# Summarize overall direct-vs-chain value for the clean analysis cohort

overall_chain_value_summary = pd.DataFrame(
    [
        {
            "cohort": "5v5_outside_origin_oz",
            "chains": len(analysis),
            "observed_goal_rate": analysis["chain_goal"].mean(),
            "mean_first_xg": analysis["first_evaluated_xg"].mean(),
            "median_first_xg": analysis["first_evaluated_xg"].median(),
            "mean_chain_max_xg": analysis["chain_max_xg"].mean(),
            "median_chain_max_xg": analysis["chain_max_xg"].median(),
            "mean_chain_sum_xg": analysis["chain_sum_xg"].mean(),
            "median_chain_sum_xg": analysis["chain_sum_xg"].median(),
            "mean_upgrade_max_minus_first_xg": analysis["upgrade_max_minus_first_xg"].mean(),
            "median_upgrade_max_minus_first_xg": analysis["upgrade_max_minus_first_xg"].median(),
            "mean_upgrade_sum_minus_first_xg": analysis["upgrade_sum_minus_first_xg"].mean(),
            "median_upgrade_sum_minus_first_xg": analysis["upgrade_sum_minus_first_xg"].median(),
            "followup_rate": analysis["has_followup_chance"].mean(),
            "deflection_rate": analysis["has_deflection"].mean(),
        }
    ]
)

overall_chain_value_summary

,cohort,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,median_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,followup_rate,deflection_rate
0,5v5_outside_origin_oz,21536,0.024935,0.016434,0.004856,0.023758,0.005535,0.025854,0.005663,0.007324,0.000000,0.009421,0.000000,0.097186,0.056974


In [16]:
# Create an easier-to-read percentage version of the overall summary

overall_chain_value_summary_display = overall_chain_value_summary.copy()

pct_columns = [
    "observed_goal_rate",
    "mean_first_xg",
    "median_first_xg",
    "mean_chain_max_xg",
    "median_chain_max_xg",
    "mean_chain_sum_xg",
    "median_chain_sum_xg",
    "mean_upgrade_max_minus_first_xg",
    "median_upgrade_max_minus_first_xg",
    "mean_upgrade_sum_minus_first_xg",
    "median_upgrade_sum_minus_first_xg",
    "followup_rate",
    "deflection_rate",
]

for col in pct_columns:
    overall_chain_value_summary_display[col] = (
        100 * overall_chain_value_summary_display[col]
    ).round(3)

overall_chain_value_summary_display

,cohort,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,median_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,followup_rate,deflection_rate
0,5v5_outside_origin_oz,21536,2.493000,1.643000,0.486000,2.376000,0.553000,2.585000,0.566000,0.732000,0.000000,0.942000,0.000000,9.719000,5.697000


## 7. Context And Slot-Support Chain-Value Decomposition

Decompose direct and chain value by tactical context and observed attacking slot support.

This is the core table for Notebook 11.

The key question is whether downstream upgrade value is concentrated in particular context/support structures.

In [17]:
# Summarize direct-vs-chain value by context and observed attacking slot support

context_slot_value_summary = (
    analysis
    .groupby(["shot_context", "observed_attackers_slot_bin"], observed=False)
    .agg(
        chains=("chain_id", "count"),
        observed_goal_rate=("chain_goal", "mean"),
        mean_first_xg=("first_evaluated_xg", "mean"),
        median_first_xg=("first_evaluated_xg", "median"),
        mean_chain_max_xg=("chain_max_xg", "mean"),
        median_chain_max_xg=("chain_max_xg", "median"),
        mean_chain_sum_xg=("chain_sum_xg", "mean"),
        median_chain_sum_xg=("chain_sum_xg", "median"),
        mean_upgrade_max_minus_first_xg=("upgrade_max_minus_first_xg", "mean"),
        median_upgrade_max_minus_first_xg=("upgrade_max_minus_first_xg", "median"),
        mean_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "mean"),
        median_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "median"),
        mean_sum_minus_max_xg=("chain_sum_minus_max_xg", "mean"),
        followup_rate=("has_followup_chance", "mean"),
        deflection_rate=("has_deflection", "mean"),
    )
    .reset_index()
)

context_slot_value_summary

,shot_context,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,median_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg,followup_rate,deflection_rate
0,other_or_unknown,0,13,0.000000,0.007096,0.005260,0.007192,0.005260,0.007428,0.005260,0.000096,0.000000,0.000333,0.000000,0.000236,0.153846,0.000000
1,other_or_unknown,1,1,1.000000,0.004350,0.004350,0.006905,0.006905,0.011255,0.011255,0.002555,0.002555,0.006905,0.006905,0.004350,1.000000,0.000000
2,other_or_unknown,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,other_or_unknown,3+,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,oz_entry_5s_proxy,0,1927,0.018682,0.011902,0.004907,0.016805,0.005550,0.018148,0.005672,0.004903,0.000000,0.006245,0.000000,0.001342,0.091853,0.031136
5,oz_entry_5s_proxy,1,1398,0.025036,0.015328,0.005738,0.022475,0.006290,0.025218,0.006451,0.007147,0.000000,0.009891,0.000000,0.002744,0.095851,0.054363
6,oz_entry_5s_proxy,2,610,0.032787,0.019710,0.006184,0.030200,0.007429,0.032178,0.007595,0.010490,0.000000,0.012468,0.000000,0.001978,0.106557,0.070492
7,oz_entry_5s_proxy,3+,85,0.011765,0.024562,0.005462,0.031362,0.005977,0.032368,0.005977,0.006800,0.000000,0.007806,0.000000,0.001005,0.105882,0.141176
8,oz_faceoff_5s,0,431,0.020882,0.014024,0.003720,0.023935,0.004440,0.024964,0.004583,0.009911,0.000000,0.010940,0.000000,0.001029,0.099768,0.039443
9,oz_faceoff_5s,1,891,0.024691,0.016229,0.004116,0.021417,0.004583,0.023483,0.004762,0.005188,0.000000,0.007254,0.000000,0.002065,0.090909,0.053872


In [18]:
# Display the main hockey contexts and suppress tiny cells for interpretation

main_contexts = [
    "oz_entry_5s_proxy",
    "oz_faceoff_5s",
    "settled_offensive_zone_proxy",
]

MIN_INTERPRETABLE_CHAINS = 100

context_slot_value_summary_main = (
    context_slot_value_summary[
        context_slot_value_summary["shot_context"].isin(main_contexts)
        & context_slot_value_summary["chains"].ge(MIN_INTERPRETABLE_CHAINS)
    ]
    .sort_values(["shot_context", "observed_attackers_slot_bin"])
    .reset_index(drop=True)
)

context_slot_value_summary_main

,shot_context,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,median_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg,followup_rate,deflection_rate
0,oz_entry_5s_proxy,0,1927,0.018682,0.011902,0.004907,0.016805,0.005550,0.018148,0.005672,0.004903,0.000000,0.006245,0.000000,0.001342,0.091853,0.031136
1,oz_entry_5s_proxy,1,1398,0.025036,0.015328,0.005738,0.022475,0.006290,0.025218,0.006451,0.007147,0.000000,0.009891,0.000000,0.002744,0.095851,0.054363
2,oz_entry_5s_proxy,2,610,0.032787,0.019710,0.006184,0.030200,0.007429,0.032178,0.007595,0.010490,0.000000,0.012468,0.000000,0.001978,0.106557,0.070492
3,oz_faceoff_5s,0,431,0.020882,0.014024,0.003720,0.023935,0.004440,0.024964,0.004583,0.009911,0.000000,0.010940,0.000000,0.001029,0.099768,0.039443
4,oz_faceoff_5s,1,891,0.024691,0.016229,0.004116,0.021417,0.004583,0.023483,0.004762,0.005188,0.000000,0.007254,0.000000,0.002065,0.090909,0.053872
5,oz_faceoff_5s,2,672,0.028274,0.016571,0.004649,0.025439,0.005524,0.027587,0.005600,0.008868,0.000000,0.011015,0.000000,0.002147,0.105655,0.065476
6,oz_faceoff_5s,3+,121,0.049587,0.017366,0.004255,0.030779,0.004970,0.032873,0.004970,0.013413,0.000000,0.015506,0.000000,0.002094,0.107438,0.057851
7,settled_offensive_zone_proxy,0,5909,0.019293,0.012747,0.004324,0.018299,0.004844,0.019646,0.004929,0.005553,0.000000,0.006899,0.000000,0.001347,0.085463,0.039601
8,settled_offensive_zone_proxy,1,5912,0.026218,0.018588,0.005061,0.026263,0.005823,0.028661,0.006010,0.007674,0.000000,0.010073,0.000000,0.002398,0.101488,0.069350
9,settled_offensive_zone_proxy,2,3145,0.034340,0.021208,0.005197,0.031467,0.006121,0.034447,0.006326,0.010259,0.000000,0.013239,0.000000,0.002980,0.110334,0.076630


In [19]:
# Create a percentage display version of the context-slot value summary

context_slot_value_summary_display = context_slot_value_summary_main.copy()

value_rate_columns = [
    "observed_goal_rate",
    "mean_first_xg",
    "median_first_xg",
    "mean_chain_max_xg",
    "median_chain_max_xg",
    "mean_chain_sum_xg",
    "median_chain_sum_xg",
    "mean_upgrade_max_minus_first_xg",
    "median_upgrade_max_minus_first_xg",
    "mean_upgrade_sum_minus_first_xg",
    "median_upgrade_sum_minus_first_xg",
    "mean_sum_minus_max_xg",
    "followup_rate",
    "deflection_rate",
]

for col in value_rate_columns:
    context_slot_value_summary_display[col] = (
        100 * context_slot_value_summary_display[col]
    ).round(3)

context_slot_value_summary_display

,shot_context,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,median_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg,followup_rate,deflection_rate
0,oz_entry_5s_proxy,0,1927,1.868000,1.190000,0.491000,1.681000,0.555000,1.815000,0.567000,0.490000,0.000000,0.625000,0.000000,0.134000,9.185000,3.114000
1,oz_entry_5s_proxy,1,1398,2.504000,1.533000,0.574000,2.247000,0.629000,2.522000,0.645000,0.715000,0.000000,0.989000,0.000000,0.274000,9.585000,5.436000
2,oz_entry_5s_proxy,2,610,3.279000,1.971000,0.618000,3.020000,0.743000,3.218000,0.759000,1.049000,0.000000,1.247000,0.000000,0.198000,10.656000,7.049000
3,oz_faceoff_5s,0,431,2.088000,1.402000,0.372000,2.394000,0.444000,2.496000,0.458000,0.991000,0.000000,1.094000,0.000000,0.103000,9.977000,3.944000
4,oz_faceoff_5s,1,891,2.469000,1.623000,0.412000,2.142000,0.458000,2.348000,0.476000,0.519000,0.000000,0.725000,0.000000,0.207000,9.091000,5.387000
5,oz_faceoff_5s,2,672,2.827000,1.657000,0.465000,2.544000,0.552000,2.759000,0.560000,0.887000,0.000000,1.102000,0.000000,0.215000,10.565000,6.548000
6,oz_faceoff_5s,3+,121,4.959000,1.737000,0.426000,3.078000,0.497000,3.287000,0.497000,1.341000,0.000000,1.551000,0.000000,0.209000,10.744000,5.785000
7,settled_offensive_zone_proxy,0,5909,1.929000,1.275000,0.432000,1.830000,0.484000,1.965000,0.493000,0.555000,0.000000,0.690000,0.000000,0.135000,8.546000,3.960000
8,settled_offensive_zone_proxy,1,5912,2.622000,1.859000,0.506000,2.626000,0.582000,2.866000,0.601000,0.767000,0.000000,1.007000,0.000000,0.240000,10.149000,6.935000
9,settled_offensive_zone_proxy,2,3145,3.434000,2.121000,0.520000,3.147000,0.612000,3.445000,0.633000,1.026000,0.000000,1.324000,0.000000,0.298000,11.033000,7.663000


## 8. Ranking Comparison: Origin xG vs Chain Value vs Observed Goals

Compare how context/support groups rank under different value definitions.

This checks whether origin-shot xG, chain-value proxies, and observed goal rates tell the same tactical story.

In [20]:
# Rank interpretable context-support groups by first xG, chain value, and observed goal rate

ranking_metrics = [
    "observed_goal_rate",
    "mean_first_xg",
    "mean_chain_max_xg",
    "mean_chain_sum_xg",
    "mean_upgrade_sum_minus_first_xg",
]

context_slot_rankings = context_slot_value_summary_main[
    [
        "shot_context",
        "observed_attackers_slot_bin",
        "chains",
        *ranking_metrics,
    ]
].copy()

for metric in ranking_metrics:
    context_slot_rankings[f"rank_{metric}"] = (
        context_slot_rankings[metric]
        .rank(ascending=False, method="min")
        .astype(int)
    )

context_slot_rankings = context_slot_rankings.sort_values(
    ["rank_observed_goal_rate", "shot_context", "observed_attackers_slot_bin"]
).reset_index(drop=True)

context_slot_rankings

,shot_context,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,mean_chain_max_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg,rank_observed_goal_rate,rank_mean_first_xg,rank_mean_chain_max_xg,rank_mean_chain_sum_xg,rank_mean_upgrade_sum_minus_first_xg
0,oz_faceoff_5s,3+,121,0.049587,0.017366,0.030779,0.032873,0.015506,1,5,3,3,1
1,settled_offensive_zone_proxy,2,3145,0.034340,0.021208,0.031467,0.034447,0.013239,2,2,2,2,3
2,oz_entry_5s_proxy,2,610,0.032787,0.019710,0.030200,0.032178,0.012468,3,3,4,4,4
3,oz_faceoff_5s,2,672,0.028274,0.016571,0.025439,0.027587,0.011015,4,6,6,6,5
4,settled_offensive_zone_proxy,1,5912,0.026218,0.018588,0.026263,0.028661,0.010073,5,4,5,5,7
5,settled_offensive_zone_proxy,3+,421,0.026128,0.023004,0.033427,0.038059,0.015055,6,1,1,1,2
6,oz_entry_5s_proxy,1,1398,0.025036,0.015328,0.022475,0.025218,0.009891,7,8,8,7,8
7,oz_faceoff_5s,1,891,0.024691,0.016229,0.021417,0.023483,0.007254,8,7,9,9,9
8,oz_faceoff_5s,0,431,0.020882,0.014024,0.023935,0.024964,0.010940,9,9,7,8,6
9,settled_offensive_zone_proxy,0,5909,0.019293,0.012747,0.018299,0.019646,0.006899,10,10,10,10,10


In [21]:
# Create a compact ranking display focused on the main contrast

context_slot_rankings_display = context_slot_rankings[
    [
        "shot_context",
        "observed_attackers_slot_bin",
        "chains",
        "observed_goal_rate",
        "mean_first_xg",
        "mean_chain_sum_xg",
        "mean_upgrade_sum_minus_first_xg",
        "rank_observed_goal_rate",
        "rank_mean_first_xg",
        "rank_mean_chain_sum_xg",
        "rank_mean_upgrade_sum_minus_first_xg",
    ]
].copy()

for col in [
    "observed_goal_rate",
    "mean_first_xg",
    "mean_chain_sum_xg",
    "mean_upgrade_sum_minus_first_xg",
]:
    context_slot_rankings_display[col] = (
        100 * context_slot_rankings_display[col]
    ).round(3)

context_slot_rankings_display

,shot_context,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg,rank_observed_goal_rate,rank_mean_first_xg,rank_mean_chain_sum_xg,rank_mean_upgrade_sum_minus_first_xg
0,oz_faceoff_5s,3+,121,4.959000,1.737000,3.287000,1.551000,1,5,3,1
1,settled_offensive_zone_proxy,2,3145,3.434000,2.121000,3.445000,1.324000,2,2,2,3
2,oz_entry_5s_proxy,2,610,3.279000,1.971000,3.218000,1.247000,3,3,4,4
3,oz_faceoff_5s,2,672,2.827000,1.657000,2.759000,1.102000,4,6,6,5
4,settled_offensive_zone_proxy,1,5912,2.622000,1.859000,2.866000,1.007000,5,4,5,7
5,settled_offensive_zone_proxy,3+,421,2.613000,2.300000,3.806000,1.505000,6,1,1,2
6,oz_entry_5s_proxy,1,1398,2.504000,1.533000,2.522000,0.989000,7,8,7,8
7,oz_faceoff_5s,1,891,2.469000,1.623000,2.348000,0.725000,8,7,9,9
8,oz_faceoff_5s,0,431,2.088000,1.402000,2.496000,1.094000,9,9,8,6
9,settled_offensive_zone_proxy,0,5909,1.929000,1.275000,1.965000,0.690000,10,10,10,10


## 9. Mechanism Decomposition: Follow-Ups And Deflections

Decompose direct-vs-chain value by realized chain mechanism.

This section separates outside-origin chains into mechanism groups:

- no follow-up and no deflection
- deflection only
- follow-up only
- both follow-up and deflection

These variables occur after the origin shot, so they are not release-time prediction features. They are used here only to explain how downstream value is realized.

In [22]:
# Classify realized downstream mechanisms for each chain

def classify_chain_mechanism(row):
    has_followup = bool(row["has_followup_chance"])
    has_deflection = bool(row["has_deflection"])

    if has_followup and has_deflection:
        return "followup_and_deflection"

    if has_followup:
        return "followup_only"

    if has_deflection:
        return "deflection_only"

    return "no_followup_no_deflection"


analysis["chain_mechanism"] = analysis.apply(classify_chain_mechanism, axis=1)

mechanism_order = [
    "no_followup_no_deflection",
    "deflection_only",
    "followup_only",
    "followup_and_deflection",
]

analysis["chain_mechanism"] = pd.Categorical(
    analysis["chain_mechanism"],
    categories=mechanism_order,
    ordered=True,
)

analysis["chain_mechanism"].value_counts(dropna=False).sort_index()

chain_mechanism
no_followup_no_deflection    18381
deflection_only               1062
followup_only                 1928
followup_and_deflection        165
Name: count, dtype: int64

In [23]:
# Summarize direct-vs-chain value by realized mechanism

mechanism_value_summary = (
    analysis
    .groupby("chain_mechanism", observed=True)
    .agg(
        chains=("chain_id", "count"),
        observed_goal_rate=("chain_goal", "mean"),
        mean_first_xg=("first_evaluated_xg", "mean"),
        median_first_xg=("first_evaluated_xg", "median"),
        mean_chain_max_xg=("chain_max_xg", "mean"),
        median_chain_max_xg=("chain_max_xg", "median"),
        mean_chain_sum_xg=("chain_sum_xg", "mean"),
        median_chain_sum_xg=("chain_sum_xg", "median"),
        mean_upgrade_max_minus_first_xg=("upgrade_max_minus_first_xg", "mean"),
        mean_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "mean"),
        mean_sum_minus_max_xg=("chain_sum_minus_max_xg", "mean"),
    )
    .reset_index()
)

mechanism_value_summary

,chain_mechanism,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg
0,no_followup_no_deflection,18381,0.011479,0.010758,0.004437,0.010758,0.004437,0.010758,0.004437,0.000000,0.000000,0.000000
1,deflection_only,1062,0.115819,0.110783,0.088983,0.110783,0.088983,0.110783,0.088983,0.000000,0.000000,0.000000
2,followup_only,1928,0.086100,0.012170,0.004838,0.084127,0.045319,0.100944,0.052930,0.071957,0.088773,0.016816
3,followup_and_deflection,165,0.224242,0.091236,0.069390,0.206380,0.155863,0.283567,0.222399,0.115144,0.192331,0.077187


In [24]:
# Create a percentage display version of the mechanism summary

mechanism_value_summary_display = mechanism_value_summary.copy()

mechanism_pct_columns = [
    "observed_goal_rate",
    "mean_first_xg",
    "median_first_xg",
    "mean_chain_max_xg",
    "median_chain_max_xg",
    "mean_chain_sum_xg",
    "median_chain_sum_xg",
    "mean_upgrade_max_minus_first_xg",
    "mean_upgrade_sum_minus_first_xg",
    "mean_sum_minus_max_xg",
]

for col in mechanism_pct_columns:
    mechanism_value_summary_display[col] = (
        100 * mechanism_value_summary_display[col]
    ).round(3)

mechanism_value_summary_display

,chain_mechanism,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_max_minus_first_xg,mean_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg
0,no_followup_no_deflection,18381,1.148000,1.076000,0.444000,1.076000,0.444000,1.076000,0.444000,0.000000,0.000000,0.000000
1,deflection_only,1062,11.582000,11.078000,8.898000,11.078000,8.898000,11.078000,8.898000,0.000000,0.000000,0.000000
2,followup_only,1928,8.610000,1.217000,0.484000,8.413000,4.532000,10.094000,5.293000,7.196000,8.877000,1.682000
3,followup_and_deflection,165,22.424000,9.124000,6.939000,20.638000,15.586000,28.357000,22.240000,11.514000,19.233000,7.719000


In [25]:
# Summarize mechanism mix by context and observed attacking slot support

mechanism_mix_by_context_slot = (
    analysis
    .groupby(
        [
            "shot_context",
            "observed_attackers_slot_bin",
            "chain_mechanism",
        ],
        observed=True,
    )
    .agg(
        chains=("chain_id", "count"),
    )
    .reset_index()
)

mechanism_mix_by_context_slot["pct_within_context_slot"] = (
    mechanism_mix_by_context_slot["chains"]
    / mechanism_mix_by_context_slot.groupby(
        ["shot_context", "observed_attackers_slot_bin"],
        observed=True,
    )["chains"].transform("sum")
)

mechanism_mix_main = (
    mechanism_mix_by_context_slot[
        mechanism_mix_by_context_slot["shot_context"].isin(main_contexts)
    ]
    .sort_values(
        [
            "shot_context",
            "observed_attackers_slot_bin",
            "chain_mechanism",
        ]
    )
    .reset_index(drop=True)
)

mechanism_mix_main

,shot_context,observed_attackers_slot_bin,chain_mechanism,chains,pct_within_context_slot
0,oz_entry_5s_proxy,0,no_followup_no_deflection,1702,0.883238
1,oz_entry_5s_proxy,0,deflection_only,48,0.024909
2,oz_entry_5s_proxy,0,followup_only,165,0.085625
3,oz_entry_5s_proxy,0,followup_and_deflection,12,0.006227
4,oz_entry_5s_proxy,1,no_followup_no_deflection,1195,0.854793
5,oz_entry_5s_proxy,1,deflection_only,69,0.049356
6,oz_entry_5s_proxy,1,followup_only,127,0.090844
7,oz_entry_5s_proxy,1,followup_and_deflection,7,0.005007
8,oz_entry_5s_proxy,2,no_followup_no_deflection,505,0.827869
9,oz_entry_5s_proxy,2,deflection_only,40,0.065574


In [26]:
# Create compact mechanism-mix table for the main contexts

mechanism_mix_compact = (
    mechanism_mix_main
    .pivot_table(
        index=["shot_context", "observed_attackers_slot_bin"],
        columns="chain_mechanism",
        values="pct_within_context_slot",
        fill_value=0,
        observed=True,
    )
    .reset_index()
)

mechanism_mix_compact_display = mechanism_mix_compact.copy()

for col in mechanism_order:
    if col in mechanism_mix_compact_display.columns:
        mechanism_mix_compact_display[col] = (
            100 * mechanism_mix_compact_display[col]
        ).round(2)

mechanism_mix_compact_display

chain_mechanism,shot_context,observed_attackers_slot_bin,no_followup_no_deflection,deflection_only,followup_only,followup_and_deflection
0,oz_entry_5s_proxy,0,88.320000,2.490000,8.560000,0.620000
1,oz_entry_5s_proxy,1,85.480000,4.940000,9.080000,0.500000
2,oz_entry_5s_proxy,2,82.790000,6.560000,10.160000,0.490000
3,oz_entry_5s_proxy,3+,77.650000,11.760000,8.240000,2.350000
4,oz_faceoff_5s,0,86.770000,3.250000,9.280000,0.700000
5,oz_faceoff_5s,1,86.200000,4.710000,8.420000,0.670000
6,oz_faceoff_5s,2,83.780000,5.650000,9.670000,0.890000
7,oz_faceoff_5s,3+,85.950000,3.310000,8.260000,2.480000
8,settled_offensive_zone_proxy,0,88.070000,3.380000,7.970000,0.580000
9,settled_offensive_zone_proxy,1,83.710000,6.140000,9.350000,0.790000


## 10. Focused Settled-OZ 2-vs-3+ Decomposition

Notebook 10 showed that the settled-OZ 2-vs-3+ pattern remains directionally consistent across basic control checks, but with uncertainty.

This section compares the two groups under direct xG, chain-sum xG, observed goal rate, and mechanism mix.

The key question is whether the 3+ group accumulates more measured chain value without matching observed goal conversion.

In [27]:
# Create focused settled-OZ 2-vs-3+ decomposition cohort

settled_oz_value = analysis[
    analysis["shot_context"].eq("settled_offensive_zone_proxy")
].copy()

settled_oz_2_vs_3_value = settled_oz_value[
    settled_oz_value["observed_attackers_slot_bin"].astype(str).isin(["2", "3+"])
].copy()

print("settled_oz_value:", settled_oz_value.shape)
print("settled_oz_2_vs_3_value:", settled_oz_2_vs_3_value.shape)

settled_oz_2_vs_3_value["observed_attackers_slot_bin"].value_counts(dropna=False)

settled_oz_value: (15387, 117)
settled_oz_2_vs_3_value: (3566, 117)


observed_attackers_slot_bin
2     3145
3+     421
0        0
1        0
Name: count, dtype: int64

In [28]:
# Summarize settled-OZ 2-vs-3+ direct-vs-chain value

settled_oz_2_vs_3_value_summary = (
    settled_oz_2_vs_3_value
    .groupby("observed_attackers_slot_bin", observed=True)
    .agg(
        chains=("chain_id", "count"),
        observed_goal_rate=("chain_goal", "mean"),
        mean_first_xg=("first_evaluated_xg", "mean"),
        median_first_xg=("first_evaluated_xg", "median"),
        mean_chain_max_xg=("chain_max_xg", "mean"),
        median_chain_max_xg=("chain_max_xg", "median"),
        mean_chain_sum_xg=("chain_sum_xg", "mean"),
        median_chain_sum_xg=("chain_sum_xg", "median"),
        mean_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "mean"),
        median_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "median"),
        mean_sum_minus_max_xg=("chain_sum_minus_max_xg", "mean"),
        followup_rate=("has_followup_chance", "mean"),
        deflection_rate=("has_deflection", "mean"),
    )
    .reset_index()
)

settled_oz_2_vs_3_value_summary

,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg,followup_rate,deflection_rate
0,2,3145,0.034340,0.021208,0.005197,0.031467,0.006121,0.034447,0.006326,0.013239,0.000000,0.002980,0.110334,0.076630
1,3+,421,0.026128,0.023004,0.005918,0.033427,0.006646,0.038059,0.006646,0.015055,0.000000,0.004632,0.106888,0.083135


In [29]:
# Create a percentage display version for settled-OZ 2-vs-3+ decomposition

settled_oz_2_vs_3_value_summary_display = settled_oz_2_vs_3_value_summary.copy()

settled_value_pct_columns = [
    "observed_goal_rate",
    "mean_first_xg",
    "median_first_xg",
    "mean_chain_max_xg",
    "median_chain_max_xg",
    "mean_chain_sum_xg",
    "median_chain_sum_xg",
    "mean_upgrade_sum_minus_first_xg",
    "median_upgrade_sum_minus_first_xg",
    "mean_sum_minus_max_xg",
    "followup_rate",
    "deflection_rate",
]

for col in settled_value_pct_columns:
    settled_oz_2_vs_3_value_summary_display[col] = (
        100 * settled_oz_2_vs_3_value_summary_display[col]
    ).round(3)

settled_oz_2_vs_3_value_summary_display

,observed_attackers_slot_bin,chains,observed_goal_rate,mean_first_xg,median_first_xg,mean_chain_max_xg,median_chain_max_xg,mean_chain_sum_xg,median_chain_sum_xg,mean_upgrade_sum_minus_first_xg,median_upgrade_sum_minus_first_xg,mean_sum_minus_max_xg,followup_rate,deflection_rate
0,2,3145,3.434000,2.121000,0.520000,3.147000,0.612000,3.445000,0.633000,1.324000,0.000000,0.298000,11.033000,7.663000
1,3+,421,2.613000,2.300000,0.592000,3.343000,0.665000,3.806000,0.665000,1.505000,0.000000,0.463000,10.689000,8.314000


In [30]:
# Compute settled-OZ 3+ minus 2 differences across direct, chain, and outcome metrics

settled_value_indexed = settled_oz_2_vs_3_value_summary.copy()
settled_value_indexed["slot_bin_str"] = (
    settled_value_indexed["observed_attackers_slot_bin"].astype(str)
)
settled_value_indexed = settled_value_indexed.set_index("slot_bin_str")

settled_oz_value_diff_3plus_minus_2 = {}

for col in settled_value_indexed.columns:
    if col in ["observed_attackers_slot_bin", "chains"]:
        continue

    settled_oz_value_diff_3plus_minus_2[col] = (
        settled_value_indexed.loc["3+", col]
        - settled_value_indexed.loc["2", col]
    )

settled_oz_value_diff_3plus_minus_2 = pd.Series(
    settled_oz_value_diff_3plus_minus_2,
    name="difference_3_plus_minus_2",
).reset_index()

settled_oz_value_diff_3plus_minus_2 = settled_oz_value_diff_3plus_minus_2.rename(
    columns={"index": "metric"}
)

settled_oz_value_diff_3plus_minus_2

,metric,difference_3_plus_minus_2
0,observed_goal_rate,-0.008212
1,mean_first_xg,0.001796
2,median_first_xg,0.000721
3,mean_chain_max_xg,0.001959
4,median_chain_max_xg,0.000524
5,mean_chain_sum_xg,0.003612
6,median_chain_sum_xg,0.000319
7,mean_upgrade_sum_minus_first_xg,0.001816
8,median_upgrade_sum_minus_first_xg,0.000000
9,mean_sum_minus_max_xg,0.001653


In [31]:
# Create a percentage display version of settled-OZ 3+ minus 2 differences

settled_oz_value_diff_3plus_minus_2_display = (
    settled_oz_value_diff_3plus_minus_2.copy()
)

settled_oz_value_diff_3plus_minus_2_display["difference_3_plus_minus_2"] = (
    100 * settled_oz_value_diff_3plus_minus_2_display["difference_3_plus_minus_2"]
).round(3)

settled_oz_value_diff_3plus_minus_2_display

,metric,difference_3_plus_minus_2
0,observed_goal_rate,-0.821000
1,mean_first_xg,0.180000
2,median_first_xg,0.072000
3,mean_chain_max_xg,0.196000
4,median_chain_max_xg,0.052000
5,mean_chain_sum_xg,0.361000
6,median_chain_sum_xg,0.032000
7,mean_upgrade_sum_minus_first_xg,0.182000
8,median_upgrade_sum_minus_first_xg,0.000000
9,mean_sum_minus_max_xg,0.165000


In [32]:
# Compare mechanism mix for settled-OZ 2 vs 3+

settled_oz_mechanism_mix_2_vs_3 = (
    settled_oz_2_vs_3_value
    .groupby(["observed_attackers_slot_bin", "chain_mechanism"], observed=True)
    .agg(
        chains=("chain_id", "count"),
    )
    .reset_index()
)

settled_oz_mechanism_mix_2_vs_3["pct_within_slot_bin"] = (
    settled_oz_mechanism_mix_2_vs_3["chains"]
    / settled_oz_mechanism_mix_2_vs_3.groupby(
        "observed_attackers_slot_bin",
        observed=True,
    )["chains"].transform("sum")
)

settled_oz_mechanism_mix_2_vs_3_display = (
    settled_oz_mechanism_mix_2_vs_3
    .pivot_table(
        index="observed_attackers_slot_bin",
        columns="chain_mechanism",
        values="pct_within_slot_bin",
        fill_value=0,
        observed=True,
    )
    .reset_index()
)

for col in mechanism_order:
    if col in settled_oz_mechanism_mix_2_vs_3_display.columns:
        settled_oz_mechanism_mix_2_vs_3_display[col] = (
            100 * settled_oz_mechanism_mix_2_vs_3_display[col]
        ).round(2)

settled_oz_mechanism_mix_2_vs_3_display

chain_mechanism,observed_attackers_slot_bin,no_followup_no_deflection,deflection_only,followup_only,followup_and_deflection
0,2,82.350000,6.610000,9.980000,1.050000
1,3+,83.140000,6.180000,8.550000,2.140000


## 11. Candidate Shoot-to-Create Profiles

Define candidate outside-shot chains where the origin shot is low-value as a terminal attempt but creates downstream chain value.

These are descriptive profiles, not causal labels.

Two candidate definitions are used:

1. Conservative shoot-to-create candidate:
   - first evaluated xG below the analysis-cohort median
   - has a follow-up chance
   - chain-sum xG exceeds first evaluated xG

2. High-upgrade low-xG candidate:
   - first evaluated xG below the analysis-cohort median
   - upgrade from first xG to chain-sum xG is in the top quartile among positive-upgrade chains

The conservative profile describes a realized follow-up mechanism. The high-upgrade profile is an ex-post upper-tail diagnostic, not a release-time shot recommendation.

In [33]:
# Define thresholds for candidate shoot-to-create profiles

first_xg_median = analysis["first_evaluated_xg"].median()

positive_upgrade_sum = analysis.loc[
    analysis["upgrade_sum_minus_first_xg"].gt(0),
    "upgrade_sum_minus_first_xg",
]

positive_upgrade_sum_75th = positive_upgrade_sum.quantile(0.75)

shoot_to_create_thresholds = {
    "first_xg_median": first_xg_median,
    "positive_upgrade_sum_minus_first_xg_75th": positive_upgrade_sum_75th,
    "positive_upgrade_chain_count": len(positive_upgrade_sum),
    "positive_upgrade_share_of_analysis": len(positive_upgrade_sum) / len(analysis),
}

shoot_to_create_thresholds

{'first_xg_median': np.float64(0.004855508683249354),
 'positive_upgrade_sum_minus_first_xg_75th': np.float64(0.12524671852588654),
 'positive_upgrade_chain_count': 2093,
 'positive_upgrade_share_of_analysis': 0.09718610698365528}

In [34]:
# Flag conservative and high-upgrade candidate shoot-to-create chains

analysis["low_origin_xg"] = analysis["first_evaluated_xg"].lt(first_xg_median)

analysis["conservative_shoot_to_create_candidate"] = (
    analysis["low_origin_xg"]
    & analysis["has_followup_chance"]
    & analysis["upgrade_sum_minus_first_xg"].gt(0)
)

analysis["high_upgrade_shoot_to_create_candidate"] = (
    analysis["low_origin_xg"]
    & analysis["upgrade_sum_minus_first_xg"].ge(positive_upgrade_sum_75th)
)

shoot_to_create_counts = pd.DataFrame(
    [
        {
            "candidate_definition": "conservative_low_xg_followup_upgrade",
            "chains": int(analysis["conservative_shoot_to_create_candidate"].sum()),
            "pct_of_analysis": analysis["conservative_shoot_to_create_candidate"].mean(),
            "observed_goal_rate": analysis.loc[
                analysis["conservative_shoot_to_create_candidate"],
                "chain_goal",
            ].mean(),
            "mean_first_xg": analysis.loc[
                analysis["conservative_shoot_to_create_candidate"],
                "first_evaluated_xg",
            ].mean(),
            "mean_chain_sum_xg": analysis.loc[
                analysis["conservative_shoot_to_create_candidate"],
                "chain_sum_xg",
            ].mean(),
            "mean_upgrade_sum_minus_first_xg": analysis.loc[
                analysis["conservative_shoot_to_create_candidate"],
                "upgrade_sum_minus_first_xg",
            ].mean(),
        },
        {
            "candidate_definition": "high_upgrade_low_xg_top_quartile_positive_upgrade",
            "chains": int(analysis["high_upgrade_shoot_to_create_candidate"].sum()),
            "pct_of_analysis": analysis["high_upgrade_shoot_to_create_candidate"].mean(),
            "observed_goal_rate": analysis.loc[
                analysis["high_upgrade_shoot_to_create_candidate"],
                "chain_goal",
            ].mean(),
            "mean_first_xg": analysis.loc[
                analysis["high_upgrade_shoot_to_create_candidate"],
                "first_evaluated_xg",
            ].mean(),
            "mean_chain_sum_xg": analysis.loc[
                analysis["high_upgrade_shoot_to_create_candidate"],
                "chain_sum_xg",
            ].mean(),
            "mean_upgrade_sum_minus_first_xg": analysis.loc[
                analysis["high_upgrade_shoot_to_create_candidate"],
                "upgrade_sum_minus_first_xg",
            ].mean(),
        },
    ]
)

shoot_to_create_counts

,candidate_definition,chains,pct_of_analysis,observed_goal_rate,mean_first_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg
0,conservative_low_xg_followup_upgrade,989,0.045923,0.073812,0.002786,0.083179,0.080393
1,high_upgrade_low_xg_top_quartile_positive_upgrade,196,0.009101,0.239796,0.002859,0.268407,0.265548


In [35]:
# Create a percentage display version of the shoot-to-create summary

shoot_to_create_counts_display = shoot_to_create_counts.copy()

shoot_to_create_pct_columns = [
    "pct_of_analysis",
    "observed_goal_rate",
    "mean_first_xg",
    "mean_chain_sum_xg",
    "mean_upgrade_sum_minus_first_xg",
]

for col in shoot_to_create_pct_columns:
    shoot_to_create_counts_display[col] = (
        100 * shoot_to_create_counts_display[col]
    ).round(3)

shoot_to_create_counts_display

,candidate_definition,chains,pct_of_analysis,observed_goal_rate,mean_first_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg
0,conservative_low_xg_followup_upgrade,989,4.592000,7.381000,0.279000,8.318000,8.039000
1,high_upgrade_low_xg_top_quartile_positive_upgrade,196,0.910000,23.980000,0.286000,26.841000,26.555000


In [36]:
# Summarize candidate shoot-to-create profiles by context and observed attacking slot support

shoot_to_create_by_context_slot = (
    analysis
    .groupby(["shot_context", "observed_attackers_slot_bin"], observed=False)
    .agg(
        chains=("chain_id", "count"),
        conservative_candidates=("conservative_shoot_to_create_candidate", "sum"),
        high_upgrade_candidates=("high_upgrade_shoot_to_create_candidate", "sum"),
        observed_goal_rate=("chain_goal", "mean"),
        mean_first_xg=("first_evaluated_xg", "mean"),
        mean_chain_sum_xg=("chain_sum_xg", "mean"),
        mean_upgrade_sum_minus_first_xg=("upgrade_sum_minus_first_xg", "mean"),
        followup_rate=("has_followup_chance", "mean"),
        deflection_rate=("has_deflection", "mean"),
    )
    .reset_index()
)

shoot_to_create_by_context_slot["conservative_candidate_rate"] = (
    shoot_to_create_by_context_slot["conservative_candidates"]
    / shoot_to_create_by_context_slot["chains"]
)

shoot_to_create_by_context_slot["high_upgrade_candidate_rate"] = (
    shoot_to_create_by_context_slot["high_upgrade_candidates"]
    / shoot_to_create_by_context_slot["chains"]
)

shoot_to_create_by_context_slot_main = (
    shoot_to_create_by_context_slot[
        shoot_to_create_by_context_slot["shot_context"].isin(main_contexts)
        & shoot_to_create_by_context_slot["chains"].ge(MIN_INTERPRETABLE_CHAINS)
    ]
    .sort_values(["shot_context", "observed_attackers_slot_bin"])
    .reset_index(drop=True)
)

shoot_to_create_by_context_slot_main

,shot_context,observed_attackers_slot_bin,chains,conservative_candidates,high_upgrade_candidates,observed_goal_rate,mean_first_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg,followup_rate,deflection_rate,conservative_candidate_rate,high_upgrade_candidate_rate
0,oz_entry_5s_proxy,0,1927,81,13,0.018682,0.011902,0.018148,0.006245,0.091853,0.031136,0.042034,0.006746
1,oz_entry_5s_proxy,1,1398,57,10,0.025036,0.015328,0.025218,0.009891,0.095851,0.054363,0.040773,0.007153
2,oz_entry_5s_proxy,2,610,28,7,0.032787,0.019710,0.032178,0.012468,0.106557,0.070492,0.045902,0.011475
3,oz_faceoff_5s,0,431,25,8,0.020882,0.014024,0.024964,0.010940,0.099768,0.039443,0.058005,0.018561
4,oz_faceoff_5s,1,891,50,11,0.024691,0.016229,0.023483,0.007254,0.090909,0.053872,0.056117,0.012346
5,oz_faceoff_5s,2,672,40,8,0.028274,0.016571,0.027587,0.011015,0.105655,0.065476,0.059524,0.011905
6,oz_faceoff_5s,3+,121,6,2,0.049587,0.017366,0.032873,0.015506,0.107438,0.057851,0.049587,0.016529
7,settled_offensive_zone_proxy,0,5909,253,42,0.019293,0.012747,0.019646,0.006899,0.085463,0.039601,0.042816,0.007108
8,settled_offensive_zone_proxy,1,5912,275,47,0.026218,0.018588,0.028661,0.010073,0.101488,0.069350,0.046516,0.007950
9,settled_offensive_zone_proxy,2,3145,157,42,0.034340,0.021208,0.034447,0.013239,0.110334,0.076630,0.049921,0.013355


In [37]:
# Create a percentage display version of the context-slot shoot-to-create table

shoot_to_create_by_context_slot_display = shoot_to_create_by_context_slot_main.copy()

shoot_to_create_context_pct_columns = [
    "conservative_candidate_rate",
    "high_upgrade_candidate_rate",
    "observed_goal_rate",
    "mean_first_xg",
    "mean_chain_sum_xg",
    "mean_upgrade_sum_minus_first_xg",
    "followup_rate",
    "deflection_rate",
]

for col in shoot_to_create_context_pct_columns:
    shoot_to_create_by_context_slot_display[col] = (
        100 * shoot_to_create_by_context_slot_display[col]
    ).round(3)

shoot_to_create_by_context_slot_display

,shot_context,observed_attackers_slot_bin,chains,conservative_candidates,high_upgrade_candidates,observed_goal_rate,mean_first_xg,mean_chain_sum_xg,mean_upgrade_sum_minus_first_xg,followup_rate,deflection_rate,conservative_candidate_rate,high_upgrade_candidate_rate
0,oz_entry_5s_proxy,0,1927,81,13,1.868000,1.190000,1.815000,0.625000,9.185000,3.114000,4.203000,0.675000
1,oz_entry_5s_proxy,1,1398,57,10,2.504000,1.533000,2.522000,0.989000,9.585000,5.436000,4.077000,0.715000
2,oz_entry_5s_proxy,2,610,28,7,3.279000,1.971000,3.218000,1.247000,10.656000,7.049000,4.590000,1.148000
3,oz_faceoff_5s,0,431,25,8,2.088000,1.402000,2.496000,1.094000,9.977000,3.944000,5.800000,1.856000
4,oz_faceoff_5s,1,891,50,11,2.469000,1.623000,2.348000,0.725000,9.091000,5.387000,5.612000,1.235000
5,oz_faceoff_5s,2,672,40,8,2.827000,1.657000,2.759000,1.102000,10.565000,6.548000,5.952000,1.190000
6,oz_faceoff_5s,3+,121,6,2,4.959000,1.737000,3.287000,1.551000,10.744000,5.785000,4.959000,1.653000
7,settled_offensive_zone_proxy,0,5909,253,42,1.929000,1.275000,1.965000,0.690000,8.546000,3.960000,4.282000,0.711000
8,settled_offensive_zone_proxy,1,5912,275,47,2.622000,1.859000,2.866000,1.007000,10.149000,6.935000,4.652000,0.795000
9,settled_offensive_zone_proxy,2,3145,157,42,3.434000,2.121000,3.445000,1.324000,11.033000,7.663000,4.992000,1.335000


## 12. Notebook 11 Computed Summary

Summarize the main direct-vs-chain value findings in computed fields.

These notes are diagnostic and should be used to guide final exhibit selection and the predictive model in Notebook 12.

In [38]:
# Build computed summary notes for Notebook 11

overall_row = overall_chain_value_summary.iloc[0]

settled_diff_lookup = settled_oz_value_diff_3plus_minus_2.set_index("metric")[
    "difference_3_plus_minus_2"
]

notebook_11_summary_notes = {
    "overall_observed_goal_rate": float(overall_row["observed_goal_rate"]),
    "overall_mean_first_xg": float(overall_row["mean_first_xg"]),
    "overall_mean_chain_sum_xg": float(overall_row["mean_chain_sum_xg"]),
    "overall_chain_sum_minus_first_xg": float(
        overall_row["mean_chain_sum_xg"] - overall_row["mean_first_xg"]
    ),
    "followup_only_observed_goal_rate": float(
        mechanism_value_summary.loc[
            mechanism_value_summary["chain_mechanism"].astype(str).eq("followup_only"),
            "observed_goal_rate",
        ].iloc[0]
    ),
    "followup_only_mean_first_xg": float(
        mechanism_value_summary.loc[
            mechanism_value_summary["chain_mechanism"].astype(str).eq("followup_only"),
            "mean_first_xg",
        ].iloc[0]
    ),
    "followup_only_mean_chain_sum_xg": float(
        mechanism_value_summary.loc[
            mechanism_value_summary["chain_mechanism"].astype(str).eq("followup_only"),
            "mean_chain_sum_xg",
        ].iloc[0]
    ),
    "settled_oz_3plus_minus_2_observed_goal_rate": float(
        settled_diff_lookup["observed_goal_rate"]
    ),
    "settled_oz_3plus_minus_2_mean_first_xg": float(
        settled_diff_lookup["mean_first_xg"]
    ),
    "settled_oz_3plus_minus_2_mean_chain_sum_xg": float(
        settled_diff_lookup["mean_chain_sum_xg"]
    ),
    "settled_oz_3plus_minus_2_followup_rate": float(
        settled_diff_lookup["followup_rate"]
    ),
    "settled_oz_3plus_minus_2_deflection_rate": float(
        settled_diff_lookup["deflection_rate"]
    ),
}

notebook_11_summary_notes

{'overall_observed_goal_rate': 0.024934992570579496,
 'overall_mean_first_xg': 0.01643352779811055,
 'overall_mean_chain_sum_xg': 0.025854486685696,
 'overall_chain_sum_minus_first_xg': 0.00942095888758545,
 'followup_only_observed_goal_rate': 0.08609958506224066,
 'followup_only_mean_first_xg': 0.012170153745399117,
 'followup_only_mean_chain_sum_xg': 0.10094357073574285,
 'settled_oz_3plus_minus_2_observed_goal_rate': -0.008211956542262536,
 'settled_oz_3plus_minus_2_mean_first_xg': 0.0017959064833563514,
 'settled_oz_3plus_minus_2_mean_chain_sum_xg': 0.003612191779015894,
 'settled_oz_3plus_minus_2_followup_rate': -0.0034455022299091087,
 'settled_oz_3plus_minus_2_deflection_rate': 0.006505821176772689}

## 13. Final Notebook Validation

Run lightweight checks before committing the notebook.

This notebook does not save a new processed dataset.

In [39]:
# Validate the final state of the direct-vs-chain decomposition notebook

final_notebook_validation = {
    "input_rows": len(chains),
    "input_unique_chain_ids": chains["chain_id"].nunique(),
    "input_duplicate_chain_ids": chains.duplicated("chain_id").sum(),
    "analysis_rows": len(analysis),
    "analysis_unique_chain_ids": analysis["chain_id"].nunique(),
    "analysis_duplicate_chain_ids": analysis.duplicated("chain_id").sum(),
    "behind_blue_line_rows_in_analysis": int(
        analysis["is_behind_offensive_blue_line_origin"].sum()
    ),
    "context_slot_value_summary_main_rows": len(context_slot_value_summary_main),
    "mechanism_value_summary_rows": len(mechanism_value_summary),
    "settled_oz_2_vs_3_value_rows": len(settled_oz_2_vs_3_value),
    "conservative_shoot_to_create_candidates": int(
        analysis["conservative_shoot_to_create_candidate"].sum()
    ),
    "high_upgrade_shoot_to_create_candidates": int(
        analysis["high_upgrade_shoot_to_create_candidate"].sum()
    ),
}

final_notebook_validation

{'input_rows': 48673,
 'input_unique_chain_ids': 48673,
 'input_duplicate_chain_ids': np.int64(0),
 'analysis_rows': 21536,
 'analysis_unique_chain_ids': 21536,
 'analysis_duplicate_chain_ids': np.int64(0),
 'behind_blue_line_rows_in_analysis': 0,
 'context_slot_value_summary_main_rows': 11,
 'mechanism_value_summary_rows': 4,
 'settled_oz_2_vs_3_value_rows': 3566,
 'conservative_shoot_to_create_candidates': 989,
 'high_upgrade_shoot_to_create_candidates': 196}

In [40]:
# Assert expected final notebook validation values

expected_final_notebook_validation = {
    "input_rows": 48673,
    "input_unique_chain_ids": 48673,
    "input_duplicate_chain_ids": 0,
    "analysis_rows": 21536,
    "analysis_unique_chain_ids": 21536,
    "analysis_duplicate_chain_ids": 0,
    "behind_blue_line_rows_in_analysis": 0,
    "context_slot_value_summary_main_rows": 11,
    "mechanism_value_summary_rows": 4,
    "settled_oz_2_vs_3_value_rows": 3566,
    "conservative_shoot_to_create_candidates": 989,
    "high_upgrade_shoot_to_create_candidates": 196,
}

for key, expected_value in expected_final_notebook_validation.items():
    actual_value = int(final_notebook_validation[key])
    assert actual_value == expected_value, (
        f"{key}: expected {expected_value}, got {actual_value}"
    )

print("Final notebook validation passed.")

Final notebook validation passed.


**Notebook 11 Decomposition Result**

This notebook supports the direct-vs-chain value framing:

- clean 5v5 outside-origin offensive-zone chains have higher observed goal rate and chain-sum xG than origin-shot xG
- follow-up chains are the clearest source of downstream upgrade value
- settled-OZ 3+ support accumulates more first xG, chain-sum xG, and deflection activity than the 2-attacker bin, but has lower observed goal conversion
- a small low-origin-xG candidate profile creates substantial downstream value, especially when follow-up chances occur

These findings motivate Notebook 12: a release-time model that predicts `chain_goal` directly and compares against SportLogiq origin-shot xG as a baseline.